In [1]:
# EZpay Payroll ETL Pipeline
#**Purpose:** Clean and validate raw EZpay CSV exports before loading into Supabase PostgreSQL database.  
#**Author:** Your Name  
#**Last Updated:** 2026-03


# EZpay Payroll ETL Pipeline

## 1. Import Libraries
## 2. Load Raw CSV
## 3. Inspect Data
## 4. Clean & Standardize
## 5. Validate Against Reference Tables
## 6. Flag Mismatches
## 7. Export to Supabase

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (2294330531.py, line 4)

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime

In [ ]:
## 1. Load Raw CSV

In [ ]:
# Update filename to match your actual EZpay export name

## file name needs to be dynamic******
RAW_FILE = 'ezpay_export.csv'
#**************************************

df_raw = pd.read_csv(RAW_FILE)

print(f"Rows loaded: {len(df_raw)}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()

In [ ]:
## 2. Inspect Data

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

In [ ]:
# Null count per column
df_raw.isnull().sum()

In [ ]:
## 3. Clean & Standardize

In [ ]:
# Work on a copy — never modify raw data
df = df_raw.copy()

# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print("Cleaned column names:")
print(list(df.columns))

In [ ]:
# Standardize text fields
df['position_name'] = df['position_name'].str.strip().str.title()
df['first_name']    = df['first_name'].str.strip().str.title()
df['last_name']     = df['last_name'].str.strip().str.title()
df['area_name']     = df['area_name'].str.strip().str.upper()

In [ ]:
# Parse dates
df['shift_date'] = pd.to_datetime(df['shift_date'], errors='coerce')

# Parse times
df['start_time'] = pd.to_datetime(df['start_time'], format='%H:%M', errors='coerce').dt.time
df['end_time']   = pd.to_datetime(df['end_time'],   format='%H:%M', errors='coerce').dt.time

In [ ]:
# Cast numeric columns
df['eid']        = pd.to_numeric(df['eid'],        errors='coerce').astype('Int64')
df['work_hours'] = pd.to_numeric(df['work_hours'], errors='coerce')
df['break_time'] = pd.to_numeric(df['break_time'], errors='coerce')
df['total_hours']= pd.to_numeric(df['total_hours'],errors='coerce')

In [ ]:
# Drop exact duplicates
before = len(df)
df = df.drop_duplicates()
print(f"Duplicate rows dropped: {before - len(df)}")
print(f"Rows remaining: {len(df)}")

In [ ]:
## 4. Validate Against Reference Tables

In [ ]:
# --- Area mapping ---
# Add all your areas here as the table grows
#***************************************************
area_mapping = {
    'FAB': 1,
    'GCS': 2,
}

#***************************************************


df['area_id'] = df['area_name'].map(area_mapping)

# Report unrecognized areas
unknown_areas = df[df['area_id'].isnull()]
if len(unknown_areas) > 0:
    print(f"WARNING: {len(unknown_areas)} rows with unrecognized area:")
    print(unknown_areas['area_name'].unique())
else:
    print("All area names matched.")

In [ ]:
# --- Position validation ---
valid_positions = ['Cleaner', 'Supervisor']  # must match your position table exactly

unknown_positions = df[~df['position_name'].isin(valid_positions)]
if len(unknown_positions) > 0:
    print(f"WARNING: {len(unknown_positions)} rows with unrecognized position:")
    print(unknown_positions['position_name'].unique())
else:
    print("All positions matched.")

In [ ]:
## 5. Flag Mismatches

In [ ]:
# Calculate expected work hours from time columns
df['calc_hours'] = (
    pd.to_datetime(df['end_time'].astype(str)) -
    pd.to_datetime(df['start_time'].astype(str))
).dt.total_seconds() / 3600 - df['break_time']

# Flag rows where stored work_hours differs from calculated by more than 0.1hr
df['hours_mismatch'] = np.abs(df['work_hours'] - df['calc_hours']) > 0.1
df['hours_missing']  = df['work_hours'].isnull()

print(f"Hours mismatches flagged: {df['hours_mismatch'].sum()}")
print(f"Hours missing flagged:    {df['hours_missing'].sum()}")

In [ ]:
# Review flagged rows before proceeding
df[df['hours_mismatch'] | df['hours_missing']]

In [ ]:
## 6. Final Clean Dataset

In [ ]:
# Drop rows missing critical fields
df_clean = df.dropna(subset=['eid', 'shift_date', 'area_id', 'position_name'])

# Drop unrecognized areas and positions
df_clean = df_clean[df_clean['area_id'].notna()]
df_clean = df_clean[df_clean['position_name'].isin(valid_positions)]

# Cast area_id to int now that nulls are removed
df_clean['area_id'] = df_clean['area_id'].astype(int)

print(f"Final clean rows ready to load: {len(df_clean)}")
df_clean.head()

In [ ]:
## 7. Export to Supabase
*Coming next — Supabase connection goes here*

In [ ]:
# Placeholder — we'll add the Supabase push here in the next step
print("Ready to load. Supabase connection coming next.")